In [8]:
%load_ext autoreload
%autoreload 3 --print --log

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# RGB-D、语义属性与二维地图

从一个合成平面开始，演示对齐输入、无效深度过滤、颜色/语义/特征关联、体素融合和保存加载。
这里的深度是光轴方向距离，正值毫米；投影坐标的 hwd.d 则是负的米制深度。

In [9]:
import numpy as np
from tempfile import TemporaryDirectory
from scipykit.task.cv import *
from graphmap.pose import Pose
from graphmap.sensor import CameraIntrinsic
from graphmap.adapters import rgbd_observation
from graphmap.environment import Environment

camera = CameraIntrinsic.from_sensor(width=64, height=48, fx=50, fy=50, ppx=32, ppy=24)
h, w = np.indices((48,64))
depth_mm = np.full((48,64), 2000, dtype=np.uint16)
depth_mm[8:16, 8:16] = 0
labels = np.where(w < 32, 1, 2).astype(np.uint32)
rgb = np.zeros((48,64,3), dtype=np.uint8)
rgb[labels == 1] = [70, 150, 220]
rgb[labels == 2] = [230, 145, 60]
features = np.stack([h / 48, w / 64], axis=-1).astype(np.float32)

In [15]:
fig, axes = subplots(1, 3, figsize=(10, 3))
for ax, data, title in zip(axes, [rgb, depth_mm, labels], ["RGB", "Axial depth (mm)", "Semantic ID"]):
    ax.imshow(data)
    ax.set(title=title, xlabel="w / column", ylabel="h / row")
disp(fig, '1')
plt.close(fig)

/gray/Storage/Repositories/graphmap/examples/assets/02_rgbd_mapping.ipynb/1.png

## 规范化观测并增量融合
`attributes` 中每张 H×W×... 图与深度图保持逐像素对齐。
输出的临时 IndexDB 以行号为键；Environment 融合后转换为世界体素键。
没有语义/颜色的数据也可以只传深度和位姿。

In [11]:
env = Environment(resolution=0.05, enable_2d_map=True, side_length=12)
for index, y in enumerate([0.0, 0.25]):
    observation = rgbd_observation(
        depth_mm, intrinsic=camera, pose_camera=Pose(y=y, z=1),
        rgb=rgb, attributes={"semantic_id": labels, "pixel_feature": features},
        depth_scale=0.001, depth_kind="axial", depth_range=(0.1,10), stride=4,
        timestamp=index * 0.1,
    )
    env.observe(**observation)
print(env.status())
print("temporary sample count:", len(observation["points_world"]))
print("unique semantic records:", env.index_db("semantic_id").size)
assert env.index_db("semantic_id").size == 2
assert env.index_db("semantic_id").num_links == env.N

{'num_points': 376, 'num_db_tables': 2, 'num_db_records': 190, 'num_db_links': 752, 'db_tables': {'pixel_feature': {'records': 188, 'links': 376}, 'semantic_id': {'records': 2, 'links': 376}}, 'frame_count': 2, 'has_2d_map': True, 'last_pose': 'Pose(pos=[0.   0.25 1.  ], euler_deg=[0. 0. 0.], scale=1) [None <- None]'}
temporary sample count: 188
unique semantic records: 2


## 回投与属性查询
`pixel_indices` 保存输出点对应的原始像素。映射后的语义需要用体素键或 `get_index_values` 查询，不读取旧的 InfoPoints.index 列。

In [12]:
hw = observation["pixel_indices"]
np.testing.assert_allclose(observation["points_hwd"][:, :2], hw, atol=1e-6)
np.testing.assert_array_equal(observation["colors"], rgb[hw[:,0],hw[:,1]])
semantic = np.array(env.get_index_values("semantic_id"))
point_keys = env.point_keys()
print("global key dtype:", point_keys.dtype)
print("first point:", env.infopoints.get_point(0))
print("first feature:", env.get_index_values("pixel_feature")[0])

global key dtype: uint64
first point: {'x': 2.0, 'y': 1.2799999713897705, 'z': 1.9600000381469727, 'r': 70, 'g': 150, 'b': 220, 'a': 255, 'timestamp_ms': 0, 'index': 0}
first feature: [0. 0.]


In [16]:
fig = plt.figure(figsize=(10, 4), dpi=120)
ax3d = fig.add_subplot(121, projection="3d")
ax3d.scatter(*env.infopoints.xyz.T, c=env.infopoints.color[:,:3] / 255, s=4)
ax3d.set(xlabel="world x (m)", ylabel="world y (m)", zlabel="world z (m)")
axmap = fig.add_subplot(122)
# Map2D 第一轴就是 world x；直接显示，不转置。
axmap.imshow(env.map2d.grid[:,:,0], origin="lower", vmin=-1, vmax=1, cmap="viridis")
axmap.set(title="Occupancy: -1 unknown, 0 obstacle", xlabel="w / world y index", ylabel="h / world x index")
fig.tight_layout()
disp(fig, '2')
plt.close(fig)

/gray/Storage/Repositories/graphmap/examples/assets/02_rgbd_mapping.ipynb/2.png

这里只把观测点投到占用栅格，没有自动识别地面或推断自由空间。下一步可以向 `Map2D` 显式提供地面点。

## 快照往返
此例使用临时目录。实际项目将路径换成自己的输出目录；快照格式含 pickle，只加载可信文件。

In [14]:
with TemporaryDirectory(prefix="graphmap-example-") as directory:
    env.save(directory)
    restored = Environment.load(directory)
    np.testing.assert_array_equal(restored.infopoints.data.view(np.uint32), env.infopoints.data.view(np.uint32))
    assert restored.get_index_values("semantic_id") == env.get_index_values("semantic_id")
    np.testing.assert_array_equal(restored.map2d.grid, env.map2d.grid)
print("point cloud, side tables and grid roundtrip passed")

point cloud, side tables and grid roundtrip passed
